# Re-run: usefulness (Paper II, RQ4)

This notebook recomputes the headline numbers of RQ4 from the committed outputs in the code package of
release `learning-graph-budget-2026`. It needs Python 3.9+ with numpy and the unzipped release (run it
from the release folder or its `notebooks/` folder, or set `RELEASE_DIR`).

**What it recomputes, and from what.** The arms were fitted on XES3G5M's learner records, which may not
be redistributed, and the graph was built by model calls whose prompts carried question text the
release does not carry, so neither can be redone from the release alone; `reproduce.py` refits the
arms for a reader who holds the corpus (steps `rq4-*`). The release does carry the frozen build itself,
one record per node, per edge and per tagged question (`studies/end-to-end/xes3g5m-build/build/`), and
each arm's area under the curve on the held-out attempts. From those the notebook:

- rechecks the build: 401 nodes, 636 edges, all between known nodes, acyclic and transitively
  reduced, and every one of the 7,652 questions tagged with one to three nodes;
- recomputes each contrast of Table 10 as the difference of the two arms' AUCs, and the verdicts from
  the pre-registered bar (a lower bound above +0.001 beats; an interval within ±0.001 of zero is
  equivalent), using the study's paired-bootstrap interval endpoints, which it does not redraw;
- recomputes the floors' range over their planted worlds, Table 11's mean of five deranged trees, and
  the coupled state's retained share as the sibling groups' gain over the coupling gain.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os
from pathlib import Path

RELEASE_ID = "learning-graph-budget-2026"   # filled in by board/render_report.py
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Code package:", shown(code_dir()))

Code package: code


In [2]:
import json
from collections import defaultdict

build = code_dir() / "studies/end-to-end/xes3g5m-build/build"
nodes = json.loads((build / "nodes.json").read_text())["nodes"]
edges = [tuple(e) for e in json.loads((build / "edges.json").read_text())["edges"]]
tags = json.loads((build / "tags.json").read_text())
ids = {n["id"] for n in nodes}
assert len(ids) == len(nodes) == 401 and len(set(edges)) == len(edges) == 636
assert all(a in ids and b in ids and a != b for a, b in edges)

succ = defaultdict(set)
for a, b in edges:
    succ[a].add(b)
def reach(a, skip=None):
    seen, stack = set(), [c for c in succ[a] if (a, c) != skip]
    while stack:
        x = stack.pop()
        if x not in seen:
            seen.add(x)
            stack += succ[x]
    return seen
assert all(a not in reach(a) for a in ids)                         # acyclic
assert all(b not in reach(a, skip=(a, b)) for a, b in edges)       # no edge is implied by a longer path
per_q = [len(t["nodes"]) for t in tags["tags"].values()]
assert len(per_q) == 7652 and not tags["untagged"] and min(per_q) == 1 and max(per_q) == 3
assert all(n in ids for t in tags["tags"].values() for n in t["nodes"])
print(f"{len(ids)} nodes, {len(edges)} edges (acyclic, transitively reduced); {len(per_q):,} questions tagged with "
      f"1 to 3 nodes ({per_q.count(1):,}, {per_q.count(2):,} and {per_q.count(3):,})")

401 nodes, 636 edges (acyclic, transitively reduced); 7,652 questions tagged with 1 to 3 nodes (2,530, 4,556 and 566)


## Table 10: the arms of the built graph

In [3]:
res = load_code_json("studies/end-to-end/xes3g5m-build/out/result.json")
B = res["B11"]
auc, diffs = B["auc"], B["differences"]
sg = lambda x, d=6: f"{x:+.{d}f}".replace("-", "−")
BAR = 0.001
PRINTED_T10 = {"M0": ("0.7298", "+0.069243 [+0.065999, +0.071845]"), "M1": ("0.7919", "+0.007209 [+0.006772, +0.007602]"),
               "M2": ("0.7965", "+0.002566 [+0.002306, +0.002789]"), "M2c": ("0.7976", None),
               "M3": ("0.7989", "+0.000161 [+0.000115, +0.000207]"), "M4": ("0.7991", None),
               "M4s": ("0.7989", "+0.000151 [+0.000107, +0.000196]"), "M4r": ("0.7991", "−0.000046 [−0.000114, +0.000021]")}
for arm, (a, d) in PRINTED_T10.items():
    got_d = None
    if d is not None:
        x = diffs[f"M4-{arm}"]
        diff = auc["M4"] - auc[arm]                    # recomputed from the two AUCs
        assert abs(diff - x["diff"]) < 1e-12
        got_d = f"{sg(diff)} [{sg(x['lo'])}, {sg(x['hi'])}]"
    print(f"{arm:4s} AUC {auc[arm]:.4f}" + (f"   M4 − {arm}: {got_d}" if got_d else ""))
    assert (f"{auc[arm]:.4f}", got_d) == (a, d), arm
beats = lambda c: diffs[c]["lo"] > BAR
assert beats("M4-M0") and beats("M4-M1") and diffs["M4-M2"]["lo"] > -0.005          # USEFUL, NOT WORSE THAN THE EXPERT TAXONOMY
assert not beats("M4-M3") and not beats("M4-M4s")                                     # EDGES ADD NOTHING
assert diffs["M4-M3"]["lo"] > 0 and diffs["M4-M3"]["hi"] < BAR                        # above zero, below the bar
assert diffs["M4-M4r"]["lo"] < 0 < diffs["M4-M4r"]["hi"] and auc["M4r"] > auc["M4"]   # the reversed graph scored slightly higher
assert (B["verdict"], B["edges_verdict"]) == ("USEFUL, NOT WORSE THAN THE EXPERT TAXONOMY", "EDGES ADD NOTHING")
m3m2 = auc["M3"] - auc["M2"]
assert sg(m3m2) == "+0.002405"
print(f"M3 − M2 {sg(m3m2)}. Verdicts: {B['verdict']}; {B['edges_verdict']}")

syn = load_code_json("studies/end-to-end/xes3g5m-build/out/synthetic.json")
planted = [w["M4-M4s"]["diff"] for w in syn["planted"]]
assert all(w["M4 beats M4s"] for w in syn["planted"]) and not any(w["M4 beats M4s"] for w in syn["null"])
assert (len(syn["planted"]), len(syn["null"]), sg(min(planted), 4), sg(max(planted), 4)) == (5, 10, "+0.0072", "+0.0101")
assert diffs["M4-M3"]["hi"] < min(planted)
print(f"Floor: M4 beat M4s in all {len(planted)} planted worlds ({sg(min(planted), 4)} to {sg(max(planted), 4)}) and in none of "
      f"the {len(syn['null'])} without; the edges' upper bound lies below the smallest planted effect.")

M0   AUC 0.7298   M4 − M0: +0.069243 [+0.065999, +0.071845]
M1   AUC 0.7919   M4 − M1: +0.007209 [+0.006772, +0.007602]
M2   AUC 0.7965   M4 − M2: +0.002566 [+0.002306, +0.002789]
M2c  AUC 0.7976
M3   AUC 0.7989   M4 − M3: +0.000161 [+0.000115, +0.000207]
M4   AUC 0.7991
M4s  AUC 0.7989   M4 − M4s: +0.000151 [+0.000107, +0.000196]
M4r  AUC 0.7991   M4 − M4r: −0.000046 [−0.000114, +0.000021]
M3 − M2 +0.002405. Verdicts: USEFUL, NOT WORSE THAN THE EXPERT TAXONOMY; EDGES ADD NOTHING
Floor: M4 beat M4s in all 5 planted worlds (+0.0072 to +0.0101) and in none of the 10 without; the edges' upper bound lies below the smallest planted effect.


## Table 11: the expert tree against trees of the same shape

In [4]:
kc = load_code_json("studies/end-to-end/xes3g5m-kc-tree-control/out/result.json")
A, D = kc["auc"], kc["differences"]
td = [A[f"Td{i}"] for i in range(5)]
fd = [A[f"Fd{i}"] for i in range(3)]
t_minus = A["T"] - sum(td) / 5
f_minus = A["F"] - sum(fd) / 3
assert abs(t_minus - D["T-Td-bar"]["diff"]) < 1e-12 and abs(f_minus - D["F-Fd-bar"]["diff"]) < 1e-12
assert abs(A["T"] - A["M2"] - D["T-M2"]["diff"]) < 1e-12 and abs(A["T"] - A["Tr"] - D["T-Tr"]["diff"]) < 1e-12
equivalent = lambda c: -BAR < D[c]["lo"] and D[c]["hi"] < BAR
PRINTED_T11 = {"T-Td-bar": "+0.000037 [+0.000010, +0.000063]", "T-M2": "+0.000051 [+0.000025, +0.000080]",
               "T-Tr": "+0.000021 [−0.000005, +0.000055]", "F-Fd-bar": "+0.000724 [+0.000638, +0.000816]"}
for c, want in PRINTED_T11.items():
    got = f"{sg(D[c]['diff'])} [{sg(D[c]['lo'])}, {sg(D[c]['hi'])}]"
    print(f"{c:9s} {got}   equivalent within ±0.001: {equivalent(c)}")
    assert got == want, c
assert equivalent("T-Td-bar") and equivalent("F-Fd-bar")      # TREE / FAMILY EQUIVALENT TO ITS DERANGEMENT
assert D["T-M2"]["lo"] < BAR                                    # TREE ADDS NOTHING (does not beat the expert KCs alone)
assert D["T-Td-bar"]["lo"] > 0 and D["F-Fd-bar"]["lo"] > 0 and D["T-Tr"]["lo"] < 0 < D["T-Tr"]["hi"]
cov = kc["coverage_test"]
assert (f"{cov['F']:.3f}", f"{min(cov[f'Fd{i}'] for i in range(3)):.3f}", f"{max(cov[f'Fd{i}'] for i in range(3)):.3f}") == ("0.519", "0.272", "0.316")
tree = kc["tree"]
assert (tree["kcs"], tree["contracted_parent_links"], tree["kcs_with_kc_parent"]) == (866, 559, 555)
print("Table 11 matches: the real tree minus the mean of five deranged trees, and the route family minus the mean of three, "
      "both within ±0.001 of zero (equivalent) though above it.")

T-Td-bar  +0.000037 [+0.000010, +0.000063]   equivalent within ±0.001: True
T-M2      +0.000051 [+0.000025, +0.000080]   equivalent within ±0.001: True
T-Tr      +0.000021 [−0.000005, +0.000055]   equivalent within ±0.001: True
F-Fd-bar  +0.000724 [+0.000638, +0.000816]   equivalent within ±0.001: True
Table 11 matches: the real tree minus the mean of five deranged trees, and the route family minus the mean of three, both within ±0.001 of zero (equivalent) though above it.


## The coupled state: how much of the coupling the expert hierarchy carries

The coupling gain is the log loss saved at a first attempt on a new skill by the fitted covariance
between skills (C2) over the same filter without it (C2-diag). The retained share is the gain of the
covariance restricted to sibling groups (C3) over C2-diag, divided by the coupling gain. The
pre-registered rule credits the hierarchy when it keeps at least half.

In [5]:
for corpus, want in (("xes", ("+0.0456", "0.141")), ("eedi", ("+0.0656", "0.348"))):
    g = load_code_json(f"studies/measurement/graph-state/out/{corpus}.json")
    gain, sib = g["C2_over_C2diag_M1"]["est"], g["C3_over_C2diag_M1"]["est"]
    share = sib / gain
    assert abs(share - g["graph_retention"]["est"]) < 1e-12
    print(f"{corpus:5s} coupling gain {sg(gain, 4)} nats, sibling groups {sg(sib, 4)}, retained share {share:.3f} -> {g['graph']}")
    assert (sg(gain, 4), f"{share:.3f}") == want
    assert share < 0.5 and g["graph"] == "EMPIRICAL COVARIANCE ONLY"
print("The coupled state's numbers match the paper.")

xes   coupling gain +0.0456 nats, sibling groups +0.0064, retained share 0.141 -> EMPIRICAL COVARIANCE ONLY
eedi  coupling gain +0.0656 nats, sibling groups +0.0228, retained share 0.348 -> EMPIRICAL COVARIANCE ONLY
The coupled state's numbers match the paper.
